# NL Energy Demand — Time Series Analysis

All time series are analysed at their **native resolution** to avoid step-function
artefacts introduced by forward-filling.

| Section | Data | Native resolution | Parquet checkpoint |
|---------|------|------------------|--------------------|
| 2 | ENTSO-E electricity load | **Hourly** | `data/processed/nl_hourly_dataset.parquet` |
| 3 | VIIRS VNP46A2 NTL | **Daily** | `data/processing_2/viirs_a2_daily/data` |
| 4 | VIIRS VNP46A1 NTL | **Daily** | `data/processing_2/viirs_a1_daily/data` |
| 5 | CBS economic & energy price indicators | **Monthly** | `data/processing_2/cbs_combined/data` |

For each series: **time series plot → STL decomposition → ACF/PACF → subseries analysis**.
Section 6 covers **multicollinearity**: Pearson heat-map, VIF, CCF and pair plot.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pyarrow.parquet as pq

warnings.filterwarnings("ignore")
%matplotlib inline

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "figure.titlesize": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
sns.set_theme(style="whitegrid", palette="tab10")

MONTH_NAMES   = ["Jan","Feb","Mar","Apr","May","Jun",
                 "Jul","Aug","Sep","Oct","Nov","Dec"]
DOW_SUN_FIRST = ["Sun","Mon","Tue","Wed","Thu","Fri","Sat"]   # day_of_week: 1=Sun
DOW_MON_FIRST = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]   # dayofweek:   0=Mon

print("Libraries loaded ✓")


In [ ]:
# Notebook lives in <repo>/analysis/  →  repo root is one level up
REPO     = Path.cwd().parent
P_FINAL  = REPO / "data" / "processed" / "nl_hourly_dataset.parquet"
P_P2_A2  = REPO / "data" / "processing_2" / "viirs_a2_daily" / "data"
P_P2_A1  = REPO / "data" / "processing_2" / "viirs_a1_daily" / "data"
P_P2_CBS = REPO / "data" / "processing_2" / "cbs_combined"   / "data"

for label, p in [
    ("Processed  (Phase 3)", P_FINAL),
    ("VIIRS A2 daily (P2)",  P_P2_A2),
    ("VIIRS A1 daily (P2)",  P_P2_A1),
    ("CBS combined  (P2)",   P_P2_CBS),
]:
    flag = "✓" if p.exists() else "✗  MISSING"
    print(f"  {flag}  {label}: {p.relative_to(REPO)}")


## 1  Data Loading


In [ ]:
# ── 1A  Phase-3 hourly dataset ──────────────────────────────────────────────
print("Loading Phase 3 hourly dataset …")
_avail = pq.read_schema(P_FINAL).names
_want  = [
    "timestamp", "entsoe_load_mw",
    "ntl_a2_mean", "ntl_a1_mean",
    "cbs_cpi_energy", "cbs_cpi_electricity", "cbs_cpi_gas",
    "cbs_gep_gas_hh_total", "cbs_gep_elec_hh_total",
    "cbs_gas_total_tax", "cbs_elec_total_tax",
    "cbs_gdp_yy", "cbs_consumption_hh_yy", "cbs_population_million",
    "year", "month", "day", "hour", "day_of_week", "is_weekend",
]
hourly = pd.read_parquet(P_FINAL, columns=[c for c in _want if c in _avail])
hourly["timestamp"] = pd.to_datetime(hourly["timestamp"])
hourly = hourly.sort_values("timestamp").set_index("timestamp")
print(f"  {len(hourly):,} rows  |  {hourly.index.min().date()} → {hourly.index.max().date()}")

# ── 1B  VIIRS A2 daily ───────────────────────────────────────────────────────
print("Loading VIIRS A2 daily …")
_vcols = ["date","ntl_mean","ntl_sum","ntl_valid_count","ntl_fill_count","ntl_invalid_count"]
viirs_a2 = pd.read_parquet(P_P2_A2, columns=_vcols)
viirs_a2["date"] = pd.to_datetime(viirs_a2["date"])
viirs_a2 = viirs_a2.sort_values("date").set_index("date")
print(f"  {len(viirs_a2):,} rows  |  {viirs_a2.index.min().date()} → {viirs_a2.index.max().date()}")

# ── 1C  VIIRS A1 daily ───────────────────────────────────────────────────────
print("Loading VIIRS A1 daily …")
viirs_a1 = pd.read_parquet(P_P2_A1, columns=_vcols)
viirs_a1["date"] = pd.to_datetime(viirs_a1["date"])
viirs_a1 = viirs_a1.sort_values("date").set_index("date")
print(f"  {len(viirs_a1):,} rows  |  {viirs_a1.index.min().date()} → {viirs_a1.index.max().date()}")

# ── 1D  CBS combined monthly ─────────────────────────────────────────────────
print("Loading CBS combined monthly …")
cbs = pd.read_parquet(P_P2_CBS)
cbs["date"] = pd.to_datetime(cbs[["year","month"]].assign(day=1))
cbs = cbs.sort_values("date").set_index("date")
print(f"  {len(cbs):,} rows  |  {cbs.index.min().date()} → {cbs.index.max().date()}")
print(f"  {len(cbs.columns)} columns: {', '.join(list(cbs.columns)[:6])} …")
print("\nAll datasets loaded ✓")


---
## 2  ENTSO-E Electricity Load  *(native resolution: hourly)*


In [ ]:
# ── 2.1  Time series evolution ───────────────────────────────────────────────
entsoe = hourly["entsoe_load_mw"].dropna()

fig, axes = plt.subplots(2, 1, figsize=(18, 9))
fig.suptitle("ENTSO-E Hourly Electricity Load — Netherlands")

axes[0].plot(entsoe.index, entsoe.values, lw=0.25, color="steelblue", alpha=0.5, label="Hourly")
roll = entsoe.rolling(24 * 30, center=True).mean()
axes[0].plot(roll.index, roll.values, color="firebrick", lw=2, label="30-day rolling mean")
axes[0].set_ylabel("Load (MW)")
axes[0].legend(fontsize=10)
axes[0].set_title("Full hourly series with 30-day rolling mean")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

grp      = entsoe.groupby(entsoe.index.year)
ann_mean = grp.mean()
ann_std  = grp.std()
dt_ann   = pd.to_datetime([f"{y}-07-01" for y in ann_mean.index])
axes[1].bar(dt_ann, ann_mean.values, width=300, color="steelblue", alpha=0.7, label="Annual mean")
axes[1].errorbar(dt_ann, ann_mean.values, yerr=ann_std.values,
                 fmt="none", color="black", capsize=4, lw=1, label="±1 std")
axes[1].set_ylabel("Load (MW)")
axes[1].set_title("Annual mean ± 1 standard deviation")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
# ── 2.2  STL decomposition  (period = 168 h = 1 week) ───────────────────────
# Use a recent 2-year window to keep runtime manageable while capturing the
# dominant weekly and daily seasonal cycles.
stl_in = entsoe.loc["2022":"2023"].asfreq("h").ffill()
PERIOD_H = 24 * 7   # 168 h: weekly cycle is the dominant seasonality for power load

stl_e = STL(stl_in, period=PERIOD_H, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(18, 12), sharex=True)
fig.suptitle("STL Decomposition — ENTSO-E Hourly Load  (period = 168 h = 1 week)")

for ax, (label, data, color) in zip(axes, [
    ("Observed",  stl_in.values,  "steelblue"),
    ("Trend",     stl_e.trend,    "firebrick"),
    ("Seasonal",  stl_e.seasonal, "seagreen"),
    ("Residual",  stl_e.resid,    "darkorange"),
]):
    ax.plot(stl_in.index, data, color=color,
            lw=0.3 if label in ("Observed", "Residual") else 1.4)
    ax.set_ylabel(label, fontsize=10)
    ax.axhline(0, color="black", lw=0.4, ls="--")

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.show()

obs_var = float(np.var(stl_in.values))
print("Variance explained (2022-2023 window):")
for name, comp in [("Trend", stl_e.trend),("Seasonal", stl_e.seasonal),("Residual", stl_e.resid)]:
    print(f"  {name:10s}: {100 * float(np.var(comp)) / obs_var:.1f}%")


In [ ]:
# ── 2.3  ACF + PACF  (max lag = 168 h = 1 week) ─────────────────────────────
# 168 lags show both the 24-h daily cycle and the 7-day weekly cycle clearly.
MAX_LAGS_H = 24 * 7

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 9))
fig.suptitle("ENTSO-E Hourly Load — ACF and PACF  (max lag = 168 h)")

plot_acf( entsoe, lags=MAX_LAGS_H, ax=ax1, alpha=0.05, title="ACF")
plot_pacf(entsoe, lags=MAX_LAGS_H, ax=ax2, alpha=0.05, method="ywm", title="PACF")

for ax in (ax1, ax2):
    ax.set_xlabel("Lag (hours)")
    for lag, lbl in [(24, "24 h\n(1 day)"), (168, "168 h\n(1 week)")]:
        ax.axvline(lag, color="firebrick", lw=1.2, ls="--", alpha=0.7)
        ax.text(lag + 1, ax.get_ylim()[1] * 0.85, lbl,
                color="firebrick", fontsize=9)

plt.tight_layout()
plt.show()

# Print top-10 ACF lags by magnitude
acf_vals = acf(entsoe.dropna(), nlags=MAX_LAGS_H, fft=True)
top10 = np.argsort(np.abs(acf_vals[1:]))[::-1][:10] + 1
print("Top-10 ACF lags by |r|:")
for lag in top10:
    d, h = divmod(int(lag), 24)
    print(f"  lag {lag:>4} h  ({d}d {h:02d}h)  r = {acf_vals[lag]:+.4f}")


In [ ]:
# ── 2.4  Subseries analysis — ENTSO-E ────────────────────────────────────────
e_df = hourly[["entsoe_load_mw","hour","day_of_week","month"]].dropna(subset=["entsoe_load_mw"])
e_df = e_df.rename(columns={"entsoe_load_mw": "load"})

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle("ENTSO-E Electricity Load — Seasonal Subseries Analysis")

# Hour of day
sns.boxplot(data=e_df, x="hour", y="load",
            ax=axes[0, 0], color="steelblue", showfliers=False, linewidth=0.7)
axes[0,0].set_title("Intra-day variation (hour of day, UTC)")
axes[0,0].set_xlabel("Hour (UTC)")
axes[0,0].set_ylabel("Load (MW)")

# Day of week  (pipeline encodes 1=Sun … 7=Sat)
sns.boxplot(data=e_df, x="day_of_week", y="load", order=list(range(1, 8)),
            ax=axes[0, 1], color="seagreen", showfliers=False, linewidth=0.7)
axes[0,1].set_title("Day-of-week variation")
axes[0,1].set_xticklabels(DOW_SUN_FIRST)
axes[0,1].set_xlabel("Day of week")
axes[0,1].set_ylabel("Load (MW)")

# Calendar month
sns.boxplot(data=e_df, x="month", y="load",
            ax=axes[1, 0], color="darkorange", showfliers=False, linewidth=0.7)
axes[1,0].set_title("Seasonal variation (calendar month)")
axes[1,0].set_xticklabels(MONTH_NAMES)
axes[1,0].set_xlabel("Month")
axes[1,0].set_ylabel("Load (MW)")

# Heat-map: day-of-week × hour
pivot = (e_df.groupby(["day_of_week","hour"])["load"]
             .median()
             .unstack("hour"))
pivot.index = DOW_SUN_FIRST
sns.heatmap(pivot, ax=axes[1, 1], cmap="YlOrRd", linewidths=0,
            cbar_kws={"label": "Median load (MW)", "shrink": 0.8})
axes[1,1].set_title("Median load: day-of-week × hour-of-day")
axes[1,1].set_xlabel("Hour (UTC)")
axes[1,1].set_ylabel("")

plt.tight_layout()
plt.show()


---
## 3  VIIRS VNP46A2 — Nighttime Light  *(native resolution: daily)*

Loaded from `data/processing_2/viirs_a2_daily/data` — **not** the hourly-upsampled
column in the final dataset. `ntl_mean` is the mean radiance over cloud-free,
quality ≤ 1 pixels in the NL bounding box (up to 629,664 pixels/day).


In [ ]:
# ── 3.1  Time series — VIIRS A2 ─────────────────────────────────────────────
a2_mean  = viirs_a2["ntl_mean"]
total_px = viirs_a2[["ntl_valid_count","ntl_fill_count","ntl_invalid_count"]].sum(axis=1)
valid_pct = viirs_a2["ntl_valid_count"].div(total_px.replace(0, np.nan)) * 100

fig, axes = plt.subplots(3, 1, figsize=(18, 11), sharex=True)
fig.suptitle("VIIRS VNP46A2 — Daily NTL Radiance, Netherlands")

axes[0].plot(a2_mean.index, a2_mean.values, lw=0.35, color="steelblue", alpha=0.65)
axes[0].plot(a2_mean.rolling(30, center=True).mean().index,
             a2_mean.rolling(30, center=True).mean().values,
             color="firebrick", lw=1.8, label="30-day rolling mean")
axes[0].set_ylabel("ntl_mean (nW/cm²/sr)")
axes[0].set_title("Mean NTL radiance of valid pixels")
axes[0].legend(fontsize=10)

axes[1].stackplot(
    viirs_a2.index,
    viirs_a2["ntl_valid_count"], viirs_a2["ntl_fill_count"], viirs_a2["ntl_invalid_count"],
    labels=["valid", "fill", "invalid"],
    colors=["seagreen", "firebrick", "darkorange"], alpha=0.7,
)
axes[1].set_ylabel("Pixel count")
axes[1].set_title("Pixel classification (max 629,664 per night)")
axes[1].legend(loc="upper right", fontsize=9)

axes[2].plot(valid_pct.index, valid_pct.values, lw=0.4, color="purple", alpha=0.7)
axes[2].plot(valid_pct.rolling(30, center=True).mean().index,
             valid_pct.rolling(30, center=True).mean().values,
             color="darkviolet", lw=1.8)
axes[2].set_ylabel("Valid pixel (%)")
axes[2].set_title("Cloud-free, quality ≤ 1 pixel fraction")
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()


In [ ]:
# ── 3.2  STL decomposition — VIIRS A2  (period = 365 days) ──────────────────
# Fill short gaps (≤ 7 consecutive days) with linear interpolation so that STL
# receives a complete, evenly-spaced series.
a2_full = a2_mean.asfreq("D").interpolate(method="linear", limit=7).dropna()

stl_a2 = STL(a2_full, period=365, seasonal=13, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(18, 12), sharex=True)
fig.suptitle("STL Decomposition — VIIRS A2 Daily NTL Mean  (period = 365 days)")

for ax, (label, data, color) in zip(axes, [
    ("Observed",  a2_full.values,  "steelblue"),
    ("Trend",     stl_a2.trend,    "firebrick"),
    ("Seasonal",  stl_a2.seasonal, "seagreen"),
    ("Residual",  stl_a2.resid,    "darkorange"),
]):
    ax.plot(a2_full.index, data, color=color,
            lw=0.35 if label in ("Observed","Residual") else 1.4)
    ax.set_ylabel(label, fontsize=10)
    ax.axhline(0, color="black", lw=0.4, ls="--")

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

obs_var = float(np.var(a2_full.values))
print("VIIRS A2 STL variance decomposition:")
for name, comp in [("Trend", stl_a2.trend),("Seasonal", stl_a2.seasonal),("Residual", stl_a2.resid)]:
    print(f"  {name:10s}: {100 * float(np.var(comp)) / obs_var:.1f}%")


In [ ]:
# ── 3.3  ACF + PACF — VIIRS A2  (max lag = 365 days) ────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 9))
fig.suptitle("VIIRS A2 Daily NTL Mean — ACF and PACF  (max lag = 365 days)")

plot_acf( a2_full, lags=365, ax=ax1, alpha=0.05, title="ACF")
plot_pacf(a2_full, lags=365, ax=ax2, alpha=0.05, method="ywm", title="PACF")

for ax in (ax1, ax2):
    ax.set_xlabel("Lag (days)")
    for lag, lbl in [(7,"7d"),(30,"30d"),(91,"~Q"),(182,"~H"),(365,"1yr")]:
        ax.axvline(lag, color="firebrick", lw=0.9, ls="--", alpha=0.6)
        ax.text(lag + 2, ax.get_ylim()[1] * 0.88, lbl, fontsize=8, color="firebrick")

plt.tight_layout()
plt.show()


In [ ]:
# ── 3.4  Subseries analysis — VIIRS A2 ──────────────────────────────────────
a2_df = viirs_a2[["ntl_mean"]].copy()
a2_df["month"]   = a2_df.index.month
a2_df["dayofwk"] = a2_df.index.dayofweek   # 0 = Monday

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("VIIRS A2 NTL Mean — Seasonal Subseries")

sns.boxplot(data=a2_df.dropna(), x="month", y="ntl_mean",
            ax=axes[0], color="steelblue", showfliers=False, linewidth=0.7)
axes[0].set_title("By calendar month")
axes[0].set_xticklabels(MONTH_NAMES, rotation=45)
axes[0].set_ylabel("ntl_mean (nW/cm²/sr)")

sns.boxplot(data=a2_df.dropna(), x="dayofwk", y="ntl_mean",
            ax=axes[1], color="darkorange", showfliers=False, linewidth=0.7)
axes[1].set_title("By day of week")
axes[1].set_xticklabels(DOW_MON_FIRST)
axes[1].set_ylabel("ntl_mean (nW/cm²/sr)")

hmap = (a2_df.dropna()
             .groupby([a2_df.dropna().index.year, "month"])["ntl_mean"]
             .mean().unstack("month"))
hmap.columns = MONTH_NAMES
sns.heatmap(hmap, ax=axes[2], cmap="YlOrRd",
            cbar_kws={"label": "Mean NTL radiance", "shrink": 0.8},
            linewidths=0.3, linecolor="white")
axes[2].set_title("Year × month heatmap")
axes[2].set_ylabel("Year")

plt.tight_layout()
plt.show()


---
## 4  VIIRS VNP46A1 — Nighttime Light  *(native resolution: daily)*

VNP46A1 (at-sensor raw radiance) undergoes the same analysis as VNP46A2.
A2 applies additional gap-filling and BRDF correction which reduces cloud
noise; comparing the two STL decompositions reveals the magnitude of that
correction.


In [ ]:
# ── 4.1  Time series — VIIRS A1 ─────────────────────────────────────────────
a1_mean  = viirs_a1["ntl_mean"]
total_a1 = viirs_a1[["ntl_valid_count","ntl_fill_count","ntl_invalid_count"]].sum(axis=1)
valid_pct_a1 = viirs_a1["ntl_valid_count"].div(total_a1.replace(0, np.nan)) * 100

fig, axes = plt.subplots(2, 1, figsize=(18, 9), sharex=True)
fig.suptitle("VIIRS VNP46A1 — Daily NTL Radiance, Netherlands")

axes[0].plot(a1_mean.index, a1_mean.values, lw=0.35, color="mediumpurple", alpha=0.65)
axes[0].plot(a1_mean.rolling(30, center=True).mean().index,
             a1_mean.rolling(30, center=True).mean().values,
             color="darkviolet", lw=1.8, label="30-day rolling mean")
axes[0].set_ylabel("ntl_mean (nW/cm²/sr)")
axes[0].set_title("Mean NTL radiance of valid pixels")
axes[0].legend(fontsize=10)

axes[1].plot(valid_pct_a1.index, valid_pct_a1.values, lw=0.4, color="mediumpurple", alpha=0.7)
axes[1].plot(valid_pct_a1.rolling(30, center=True).mean().index,
             valid_pct_a1.rolling(30, center=True).mean().values,
             color="darkviolet", lw=1.8)
axes[1].set_ylabel("Valid pixel (%)")
axes[1].set_title("Cloud-free, quality ≤ 1 pixel fraction")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()


In [ ]:
# ── 4.2  STL decomposition — VIIRS A1  (period = 365 days) ──────────────────
a1_full = a1_mean.asfreq("D").interpolate(method="linear", limit=7).dropna()

stl_a1 = STL(a1_full, period=365, seasonal=13, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(18, 12), sharex=True)
fig.suptitle("STL Decomposition — VIIRS A1 Daily NTL Mean  (period = 365 days)")

for ax, (label, data, color) in zip(axes, [
    ("Observed",  a1_full.values,  "mediumpurple"),
    ("Trend",     stl_a1.trend,    "darkviolet"),
    ("Seasonal",  stl_a1.seasonal, "seagreen"),
    ("Residual",  stl_a1.resid,    "darkorange"),
]):
    ax.plot(a1_full.index, data, color=color,
            lw=0.35 if label in ("Observed","Residual") else 1.4)
    ax.set_ylabel(label, fontsize=10)
    ax.axhline(0, color="black", lw=0.4, ls="--")

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

obs_var = float(np.var(a1_full.values))
print("VIIRS A1 STL variance decomposition:")
for name, comp in [("Trend", stl_a1.trend),("Seasonal", stl_a1.seasonal),("Residual", stl_a1.resid)]:
    print(f"  {name:10s}: {100 * float(np.var(comp)) / obs_var:.1f}%")


In [ ]:
# ── 4.3  ACF + PACF — VIIRS A1  (max lag = 365 days) ────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 9))
fig.suptitle("VIIRS A1 Daily NTL Mean — ACF and PACF  (max lag = 365 days)")

plot_acf( a1_full, lags=365, ax=ax1, alpha=0.05, title="ACF")
plot_pacf(a1_full, lags=365, ax=ax2, alpha=0.05, method="ywm", title="PACF")

for ax in (ax1, ax2):
    ax.set_xlabel("Lag (days)")
    for lag, lbl in [(7,"7d"),(30,"30d"),(91,"~Q"),(182,"~H"),(365,"1yr")]:
        ax.axvline(lag, color="darkviolet", lw=0.9, ls="--", alpha=0.6)
        ax.text(lag + 2, ax.get_ylim()[1] * 0.88, lbl, fontsize=8, color="darkviolet")

plt.tight_layout()
plt.show()


In [ ]:
# ── 4.4  Subseries analysis — VIIRS A1 ──────────────────────────────────────
a1_df = viirs_a1[["ntl_mean"]].copy()
a1_df["month"]   = a1_df.index.month
a1_df["dayofwk"] = a1_df.index.dayofweek

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("VIIRS A1 NTL Mean — Seasonal Subseries")

sns.boxplot(data=a1_df.dropna(), x="month", y="ntl_mean",
            ax=axes[0], color="mediumpurple", showfliers=False, linewidth=0.7)
axes[0].set_title("By calendar month")
axes[0].set_xticklabels(MONTH_NAMES, rotation=45)
axes[0].set_ylabel("ntl_mean (nW/cm²/sr)")

sns.boxplot(data=a1_df.dropna(), x="dayofwk", y="ntl_mean",
            ax=axes[1], color="darkviolet", showfliers=False, linewidth=0.7)
axes[1].set_title("By day of week")
axes[1].set_xticklabels(DOW_MON_FIRST)

hmap_a1 = (a1_df.dropna()
                .groupby([a1_df.dropna().index.year, "month"])["ntl_mean"]
                .mean().unstack("month"))
hmap_a1.columns = MONTH_NAMES
sns.heatmap(hmap_a1, ax=axes[2], cmap="Purples",
            cbar_kws={"label": "Mean NTL radiance", "shrink": 0.8},
            linewidths=0.3, linecolor="white")
axes[2].set_title("Year × month heatmap")

plt.tight_layout()
plt.show()


In [ ]:
# ── 4.5  A1 vs A2 comparison ─────────────────────────────────────────────────
shared = a2_mean.index.intersection(a1_mean.index)
roll_a2 = a2_mean.reindex(shared).rolling(30, center=True).mean()
roll_a1 = a1_mean.reindex(shared).rolling(30, center=True).mean()

fig, axes = plt.subplots(2, 1, figsize=(18, 9), sharex=True)
fig.suptitle("VIIRS A1 vs A2 — Comparison of 30-day Rolling Mean")

axes[0].plot(roll_a2.index, roll_a2.values, color="steelblue",    lw=1.4, label="A2  (gap-filled, BRDF-corrected)")
axes[0].plot(roll_a1.index, roll_a1.values, color="mediumpurple", lw=1.4, label="A1  (at-sensor raw)")
axes[0].set_ylabel("30-day mean ntl_mean")
axes[0].legend(fontsize=10)
axes[0].set_title("30-day rolling mean overlay")

ratio = a1_mean.reindex(shared) / a2_mean.reindex(shared).replace(0, np.nan)
axes[1].plot(ratio.index, ratio.rolling(30, center=True).mean().values,
             color="darkorange", lw=1.4)
axes[1].axhline(1.0, color="black", lw=0.8, ls="--", alpha=0.5)
axes[1].set_ylabel("A1 / A2  ratio")
axes[1].set_title("A1-to-A2 ratio of mean NTL (30-day rolling)")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()


---
## 5  CBS Economic & Energy Price Indicators  *(native resolution: monthly)*

Analysed at native monthly resolution from `data/processing_2/cbs_combined/data`
to avoid the step-function artefacts from forward-filling into the hourly dataset.

| Group | Key columns | Coverage |
|-------|-------------|----------|
| Energy CPI | `cbs_cpi_energy/electricity/gas` | 1996–2025 |
| Gas & Elec. Prices (GEP) | `cbs_gep_gas_hh_total`, `cbs_gep_elec_hh_total` | 2009–2025 |
| Consumer tariffs | `cbs_gas_total_tax`, `cbs_elec_total_tax` | 2018–2026 |
| Macroeconomic | `cbs_gdp_yy`, `cbs_consumption_hh_yy` | 1996–2025 |


In [ ]:
# ── 5.1  CBS time series evolution ───────────────────────────────────────────
cbs_groups = {
    "Energy CPI  (2015 = 100)": {
        "cols":   ["cbs_cpi_energy","cbs_cpi_electricity","cbs_cpi_gas"],
        "ylabel": "Price index",
    },
    "Gas & Electricity Prices — household band": {
        "cols":   ["cbs_gep_gas_hh_total","cbs_gep_elec_hh_total",
                   "cbs_gep_gas_hh_supply","cbs_gep_elec_hh_supply"],
        "ylabel": "€/m³ or €/kWh",
    },
    "Consumer Energy Tariffs": {
        "cols":   ["cbs_gas_total_tax","cbs_elec_total_tax",
                   "cbs_gas_transport_rate","cbs_elec_transport_rate"],
        "ylabel": "€/unit or €/year",
    },
    "Macroeconomic — year-over-year volume change (%)": {
        "cols":   ["cbs_gdp_yy","cbs_consumption_hh_yy",
                   "cbs_exports_total_yy","cbs_imports_total_yy"],
        "ylabel": "% change (y/y)",
    },
}

fig, axes = plt.subplots(len(cbs_groups), 1, figsize=(18, 5 * len(cbs_groups)))
fig.suptitle("CBS Monthly Indicator Groups — Time Series Evolution", y=1.01)

for ax, (grp, cfg) in zip(axes, cbs_groups.items()):
    for col in [c for c in cfg["cols"] if c in cbs.columns]:
        s = cbs[col].dropna()
        ax.plot(s.index, s.values, lw=1.6, label=col.replace("cbs_",""))
    ax.set_title(grp)
    ax.set_ylabel(cfg["ylabel"])
    ax.axhline(0, color="black", lw=0.5, ls="--", alpha=0.4)
    ax.legend(loc="upper left", fontsize=9, ncol=2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()


In [ ]:
# ── 5.2  STL decomposition — key CBS monthly series  (period = 12 months) ────
stl_cbs_targets = [
    ("cbs_cpi_energy",       "Energy CPI",       "steelblue"),
    ("cbs_gep_gas_hh_total", "GEP gas HH total", "firebrick"),
    ("cbs_gdp_yy",           "GDP y/y change",   "seagreen"),
]

for col, label, color in stl_cbs_targets:
    if col not in cbs.columns:
        print(f"Skipping {col} — not found.")
        continue
    s = cbs[col].dropna().asfreq("MS")
    if len(s) < 36:
        print(f"Skipping {col} — insufficient data ({len(s)} rows).")
        continue
    stl_c = STL(s, period=12, seasonal=5, robust=True).fit()

    fig, axes = plt.subplots(4, 1, figsize=(18, 10), sharex=True)
    fig.suptitle(f"STL Decomposition — {label}  (period = 12 months)")
    for ax, (lbl, data) in zip(axes, [
        ("Observed", s.values), ("Trend", stl_c.trend),
        ("Seasonal", stl_c.seasonal), ("Residual", stl_c.resid),
    ]):
        ax.plot(s.index, data, color=color,
                lw=0.5 if lbl in ("Observed","Residual") else 1.4)
        ax.set_ylabel(lbl, fontsize=10)
        ax.axhline(0, color="black", lw=0.4, ls="--")
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.tight_layout()
    plt.show()

    obs_var = float(np.var(s.values))
    print(f"{label}:  Trend {100*np.var(stl_c.trend)/obs_var:.1f}%  |  "
          f"Seasonal {100*np.var(stl_c.seasonal)/obs_var:.1f}%  |  "
          f"Residual {100*np.var(stl_c.resid)/obs_var:.1f}%")


In [ ]:
# ── 5.3  ACF + PACF — CBS monthly series  (max lag = 36 months) ──────────────
MAX_LAGS_MONTHLY = 36

acf_cbs = [
    ("cbs_cpi_energy",        "Energy CPI",                "steelblue"),
    ("cbs_gep_gas_hh_total",  "GEP gas HH total",          "firebrick"),
    ("cbs_gdp_yy",            "GDP y/y",                   "seagreen"),
    ("cbs_consumption_hh_yy", "Household consumption y/y", "darkorange"),
]
acf_cbs = [(col, lbl, clr) for col, lbl, clr in acf_cbs if col in cbs.columns]

fig, axes = plt.subplots(len(acf_cbs), 2, figsize=(18, 4 * len(acf_cbs)))
fig.suptitle("CBS Monthly Series — ACF and PACF  (max lag = 36 months)", y=1.01)

for row, (col, label, color) in enumerate(acf_cbs):
    s = cbs[col].dropna()
    if len(s) <= MAX_LAGS_MONTHLY + 5:
        continue
    plot_acf( s, lags=MAX_LAGS_MONTHLY, ax=axes[row,0], alpha=0.05,
             title=f"ACF — {label}")
    plot_pacf(s, lags=MAX_LAGS_MONTHLY, ax=axes[row,1], alpha=0.05,
             method="ywm", title=f"PACF — {label}")
    for ax in axes[row]:
        ax.set_xlabel("Lag (months)")
        ax.axvline(12, color="firebrick", lw=1.2, ls="--", alpha=0.6)
        ax.axvline(24, color="firebrick", lw=0.8, ls=":",  alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
# ── 5.4  Subseries — CBS seasonal variation by calendar month ─────────────────
cbs["_cal_month"] = cbs.index.month

subseries = [
    ("cbs_cpi_energy",        "Energy CPI",        "steelblue"),
    ("cbs_gep_gas_hh_total",  "GEP gas HH total",  "firebrick"),
    ("cbs_gep_elec_hh_total", "GEP elec HH total", "darkorchid"),
    ("cbs_gdp_yy",            "GDP y/y",           "seagreen"),
]
subseries = [(c, l, clr) for c, l, clr in subseries if c in cbs.columns]

fig, axes = plt.subplots(1, len(subseries), figsize=(5 * len(subseries), 5))
fig.suptitle("CBS Monthly Series — Seasonal Variation by Calendar Month")

for ax, (col, label, color) in zip(axes, subseries):
    data = cbs[["_cal_month", col]].dropna()
    mu  = data.groupby("_cal_month")[col].mean()
    sig = data.groupby("_cal_month")[col].std()
    ax.fill_between(mu.index, mu - sig, mu + sig, alpha=0.2, color=color)
    ax.plot(mu.index, mu.values, "-o", color=color, lw=2, ms=6)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Month")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels([m[:3] for m in MONTH_NAMES], rotation=45)
    ax.axhline(float(mu.mean()), color="black", lw=0.8, ls="--", alpha=0.4)

plt.tight_layout()
plt.show()


---
## 6  Multicollinearity & Cross-Correlation Analysis

Working with the **fully merged Phase 3 hourly dataset** where all series share
a common hourly index.  CBS and VIIRS columns are forward-filled (step functions),
which inflates VIF mechanically within each period — interpret VIF in terms of
between-period variation.

| Analysis | What it measures |
|----------|-----------------|
| Pearson heat-map | Linear pairwise associations |
| VIF | Multi-predictor collinearity severity |
| CCF | Temporal lead/lag between ENTSOE load and each feature |
| Pair plot | Joint distributions and scatter (sampled) |


In [ ]:
# ── 6.1  Assemble feature matrix from Phase 3 output ─────────────────────────
_feat_want = [
    "entsoe_load_mw",
    "ntl_a2_mean", "ntl_a1_mean",
    "cbs_cpi_energy", "cbs_cpi_electricity", "cbs_cpi_gas",
    "cbs_gep_gas_hh_total", "cbs_gep_elec_hh_total",
    "cbs_gas_total_tax", "cbs_elec_total_tax",
    "cbs_gdp_yy", "cbs_consumption_hh_yy", "cbs_population_million",
]
feat = hourly[[c for c in _feat_want if c in hourly.columns]].copy()

print(f"Feature matrix : {feat.shape[0]:,} rows × {feat.shape[1]} columns")
print(f"Date range     : {feat.index.min().date()} → {feat.index.max().date()}")
print("\nNull counts per column:")
print(feat.isnull().sum().to_string())


In [ ]:
# ── 6.2  Pearson correlation heat-map ─────────────────────────────────────────
corr = feat.corr(method="pearson")

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    corr, ax=ax,
    annot=True, fmt=".2f", annot_kws={"size": 8},
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.4, linecolor="white",
    cbar_kws={"shrink": 0.8, "label": "Pearson r"},
)
ax.set_title("Pearson Correlation Matrix — All Features (hourly aligned)", fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

# Top correlated pairs (lower triangle only)
mask = np.tril(np.ones(corr.shape), k=-1).astype(bool)
pairs = (corr.where(mask)
             .stack()
             .reset_index()
             .rename(columns={"level_0": "var1", "level_1": "var2", 0: "r"}))
pairs = pairs.reindex(pairs["r"].abs().sort_values(ascending=False).index)
print("\nTop-10 most correlated feature pairs:")
for _, row in pairs.head(10).iterrows():
    print(f"  {row.var1:35s} ↔ {row.var2:35s}  r = {row.r:+.4f}")


In [ ]:
# ── 6.3  Variance Inflation Factor (VIF) ──────────────────────────────────────
X = feat.drop(columns=["entsoe_load_mw"]).dropna()
print(f"VIF input: {len(X):,} complete rows (all features non-null)")

X_vals = X.values.astype(float)
vif = pd.DataFrame({
    "Feature": X.columns,
    "VIF":     [variance_inflation_factor(X_vals, i) for i in range(X_vals.shape[1])],
}).sort_values("VIF", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 0.55 * len(vif) + 1.5))
bar_colors = [
    "firebrick"  if v > 10 else
    "darkorange" if v > 5  else "steelblue"
    for v in vif["VIF"]
]
ax.barh(vif["Feature"], vif["VIF"], color=bar_colors)
ax.axvline(5,  color="darkorange", lw=1.5, ls="--", label="VIF = 5  (moderate)")
ax.axvline(10, color="firebrick",  lw=1.5, ls="--", label="VIF = 10 (high)")
ax.set_xlabel("VIF")
ax.set_title("Variance Inflation Factor — Feature Multicollinearity Diagnostic")
ax.legend(fontsize=10)
ax.invert_yaxis()
ax.set_xlim(0, min(float(vif["VIF"].max()) * 1.1, 200))
plt.tight_layout()
plt.show()

print(vif.to_string(index=False))


In [ ]:
# ── 6.4  Cross-Correlation Function (CCF)  with ENTSOE load ──────────────────
def _ccf(x: np.ndarray, y: np.ndarray, max_lag: int):
    # Normalized cross-correlation at lags -max_lag ... +max_lag.
    x_z = (x - x.mean()) / (x.std() + 1e-12)
    y_z = (y - y.mean()) / (y.std() + 1e-12)
    full = np.correlate(x_z, y_z, mode="full") / len(x_z)
    mid  = len(full) // 2
    lags = np.arange(-max_lag, max_lag + 1)
    return lags, full[mid - max_lag: mid + max_lag + 1]

MAX_LAG_CCF  = 24 * 7  # ±168 hours
ccf_features = [c for c in feat.columns if c != "entsoe_load_mw"]
n_cols = 3
n_rows = (len(ccf_features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3.5 * n_rows), sharex=True)
fig.suptitle("Cross-Correlation with ENTSOE Load  (±168 h = ±1 week)", y=1.01)
flat = axes.flatten()

for i, col in enumerate(ccf_features):
    ax = flat[i]
    overlap = feat[["entsoe_load_mw", col]].dropna()
    if len(overlap) < MAX_LAG_CCF * 3:
        ax.text(0.5, 0.5, f"Insufficient overlap ({len(overlap)} rows)",
                transform=ax.transAxes, ha="center", va="center", fontsize=9)
        ax.set_title(col.replace("cbs_",""), fontsize=9)
        continue
    lags, r = _ccf(overlap["entsoe_load_mw"].values, overlap[col].values, MAX_LAG_CCF)
    ax.plot(lags, r, lw=1.0, color="steelblue")
    ax.axhline(0, color="black", lw=0.5)
    ax.axvline(0, color="firebrick", lw=0.8, ls="--", alpha=0.6)
    conf = 1.96 / np.sqrt(len(overlap))
    ax.fill_between(lags, -conf, conf, alpha=0.15, color="gray")
    ax.set_title(col.replace("cbs_",""), fontsize=9)
    ax.set_ylim(-1, 1)
    peak_i = int(np.argmax(np.abs(r)))
    ax.annotate(f"{lags[peak_i]:+d}h  r={r[peak_i]:.2f}",
                xy=(lags[peak_i], r[peak_i]),
                xytext=(0, 10), textcoords="offset points",
                ha="center", fontsize=7.5, color="firebrick")

for j in range(i + 1, len(flat)):
    flat[j].set_visible(False)
for ax in flat[:len(ccf_features)]:
    ax.set_xlabel("Lag (hours)", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# ── 6.5  Pair plot — compact feature subset ───────────────────────────────────
pp_cols = [c for c in [
    "entsoe_load_mw", "ntl_a2_mean",
    "cbs_cpi_energy", "cbs_gep_gas_hh_total",
    "cbs_gdp_yy",     "cbs_population_million",
] if c in feat.columns]

pp_df = feat[pp_cols].dropna()
if len(pp_df) > 5000:
    pp_df = pp_df.sample(5000, random_state=42)

g = sns.pairplot(pp_df, diag_kind="kde",
                 plot_kws={"alpha": 0.25, "s": 6, "rasterized": True},
                 diag_kws={"fill": True})
g.figure.suptitle(f"Pair Plot — Key Feature Subset  (n = {len(pp_df):,} rows sampled)",
                  y=1.02, fontsize=12)
plt.tight_layout()
plt.show()


---
## Summary

| Series | STL: dominant component | Key ACF peaks | Collinearity note |
|--------|------------------------|---------------|-------------------|
| ENTSOE hourly load | Seasonal (weekly + daily) | 24 h, 168 h | Target variable |
| VIIRS A2 daily NTL | Seasonal (annual) + trend | 365 d | Moderate corr. with A1 |
| VIIRS A1 daily NTL | Seasonal (annual) + trend | 365 d | Moderate corr. with A2 |
| CBS CPI energy | Trend dominant | 12, 24 months | High corr. with GEP |
| CBS GEP gas HH | Trend dominant | 12, 24 months | High corr. with CPI & tariffs |
| CBS GDP y/y | Seasonal (annual) | 12 months | Low VIF relative to price series |
| CBS population | Monotone trend | Very long (annual) | Proxy for time trend |

> **Note on upsampling artefacts**: In the Phase 3 hourly dataset, CBS and VIIRS
> columns are forward-filled — values are identical within each month/day respectively.
> VIF values therefore exceed what would be observed at native resolution.  For
> model-input collinearity diagnostics, prefer repeating the VIF computation on the
> native-resolution datasets loaded in Sections 3–5.
